## Care home identification notebook


this notebook is the latest version as of 17/08/2024 for identifying care home residents in connected bradford after some issues wih previous scripts were identified. it is the number that will be used in the final analysis, having it in a jupytner notebook format also enabls it to be easily upliaded to github to be made freely available.

In [33]:
library(bigrquery)
library(tidyverse) 

In [34]:
ProjectId = "yhcr-prd-bradfor-bia-core"

targetdb1 <- "yhcr-prd-bradfor-bia-core.CB_2172"
targetdb1 <- gsub(" ","",targetdb1)

targetdb2 <- 'yhcr-prd-bradfor-bia-core.CB_FDM_DeathCertificates'
targetdb2 <- gsub(" ","",targetdb2)

targetdb3 <- "yhcr-prd-bradfor-bia-core.CB_FDM_AdultSocialCare"
targetdb3 <- gsub(" ","",targetdb3)



In [35]:
#first query selects all individuals admitted to a care home at any point no age or date restrictions

In [36]:
sql1 <- 
paste(
    #updated care home cohort code, uses analytic functions so is tidier, this is the currnet query used to define the cohort,
    'with a  as (
SELECT person_id, 
tbl_adultsocialcare_services_start_date, 
servicetype, 
FinalisedPSR, 
row_number() over
(partition by person_id order by tbl_adultsocialcare_services_start_date) as care_seq 
from ',targetdb3,'.tbl_adultsocialcare_services 
where ServiceType in("Nursing","Residential") 
),
b as (
select 
* 
from a
where care_seq = 1
order by person_id, tbl_adultsocialcare_services_start_date),

c as 
(SELECT person_id,
tbl_adultsocialcare_services_end_date, 
servicetype, 
row_number() over (partition by person_id order by tbl_adultsocialcare_services_end_date desc) as care_end_seq 
from ',targetdb3,'.tbl_adultsocialcare_services 
where ServiceType in("Nursing","Residential") )
,

d as (select
* from c 
where care_end_seq = 1),

e as (
select 
b.person_id,
b.tbl_adultsocialcare_services_start_date as first_episodestartdate,
d.tbl_adultsocialcare_services_end_date as last_episodestopdate,
b.ServiceType as admission_service,
b.finalisedPSR
from b 
LEFT join d on
b.person_id= d.person_id
order by person_id),

f as (
Select 
e.*,
datetime_diff(e.last_episodestopdate,e.first_episodestartdate,day) as admission_length
from e
),

h as (
SELECT
f.person_id,
f.admission_length,
f.admission_service,
f.first_episodestartdate,
f.last_episodestopdate,
f.finalisedPSR,
ethnicity.Ethnic_Group,
g.birth_datetime,
FORMAT_DATETIME("%d-%b-%y",f.first_episodestartdate) as date_of_admission,
g.death_datetime,
g.gender_source_value as gender
FROM f 
LEFT JOIN ',targetdb3,'.person g on
f.person_id = g.person_id
LEFT JOIN ',targetdb1,'.care_home_cohort_v1_ethnicity ethnicity on 
f.person_id = ethnicity.person_id
 ),

g as (
select
DATE_DIFF(first_episodestartdate, birth_datetime, YEAR) as age_admission,
h.*,
from h
where first_episodestartdate < "2019-12-31T00:00:00"
order by age_admission desc)

select
* from g

;',sep="")

sql_tbl <-bq_project_query(ProjectId,sql1)
sql_data <- bq_table_download(sql_tbl)


In [37]:
sql_data

age_admission,person_id,admission_length,admission_service,first_episodestartdate,last_episodestopdate,finalisedPSR,Ethnic_Group,birth_datetime,date_of_admission,death_datetime,gender
<int>,<int>,<int>,<chr>,<dttm>,<dttm>,<chr>,<chr>,<dttm>,<chr>,<dttm>,<chr>
110,13165651,70,Residential,2017-10-31,2018-01-09,Physical Support - Personal Care Support,Missing,1907-09-15,31-Oct-17,2018-01-15,F
108,12773259,10,Nursing,2019-04-05,2019-04-15,Physical Support - Personal Care Support,NA,1911-07-15,05-Apr-19,2019-04-15,F
107,13380980,635,Residential,2016-04-04,2017-12-30,Physical Support - Personal care support,NA,1909-09-15,04-Apr-16,2017-12-15,F
106,13455093,2393,Residential,2016-08-22,2023-03-12,Physical Support - Personal Care Support,NA,1910-11-15,22-Aug-16,2023-03-15,F
106,13129734,974,Residential,2016-04-04,2018-12-04,Physical Support - Personal care support,NA,1910-07-15,04-Apr-16,2018-12-15,F
105,12733893,270,Nursing,2016-07-25,2017-04-21,NHS Funded,NA,1911-02-15,25-Jul-16,2017-04-15,M
105,12471351,604,Nursing,2016-04-04,2017-11-29,Physical Support - Personal care support,NA,1911-08-15,04-Apr-16,2017-11-15,F
105,12993099,116,Nursing,2018-07-20,2018-11-13,Physical Support - Personal Care Support,White,1913-03-15,20-Jul-18,2018-11-15,F
104,12696288,671,Residential,2019-08-23,2021-06-24,Physical Support - Personal Care Support,White,1915-11-15,23-Aug-19,2021-06-15,F


In [38]:
#convert vectors to date

In [39]:
sql_data$death_date <- as_date(sql_data$death_datetime)
sql_data <- sql_data %>% select(-death_datetime)

In [40]:
sql_data$episodestartdate <- as_date(sql_data$first_episodestartdate)
sql_data$episodestopdate <- as_date(sql_data$last_episodestopdate)

In [41]:
sql_data$birth_date <- as_date(sql_data$birth_datetime)
sql_data <- sql_data %>% select(-birth_datetime)

In [42]:
# remove those inidividuals admitted on dates that were identified as being erroneous
# (i.e. these individuals were actually admitted into a care home prior to 2016, but had no start date so the date was put as the date of upload, creating some skew) 
# this has been confirmed with the individual who built the social care FDM in Connected bradford

In [43]:
sql_data <- sql_data %>% filter(!(episodestartdate == "2016-04-04"))%>% filter(!(episodestartdate == "2016-07-25"))%>% filter(!(episodestartdate == "2016-08-22"))%>% filter(!(episodestartdate == "2015-11-02"))

In [44]:
sql_data %>% n_distinct('person_id')

[1] 3918

In [45]:
#how many aged less than 65 

In [46]:
sql_data_below_65 <- sql_data %>% filter(age_admission < 65)
n_distinct(sql_data_below_65$person_id)

[1] 333

In [47]:
sql_data_above_sixtyfive <- sql_data %>% filter(age_admission >= 65) 
sql_data_above_sixtyfive %>% n_distinct()


[1] 3585

In [48]:
#here we bring in the mortality data to reduce number of missig death dates 

In [49]:
sql_death <- paste('
with a as (SELECT person_id,
dod,
tbl_NEC_Deaths,
reg_date,
rank() over(partition by person_id order by dod desc, reg_date desc) as rank_seq
from ',targetdb2,'.tbl_NEC_Deaths
where person_id in (SELECT person_id from ',targetdb3,'.person where death_datetime is null)),

b as (
select
*,
row_number() over(partition by person_id order by person_id) as rank_seq_2
from a 
where rank_seq = 1)

select 
person_id, dod,reg_date from b 
where rank_seq_2 = 1 
order by person_id 
;',sep="")

tbl_death <-bq_project_query(ProjectId,sql_death)
ch_death<- bq_table_download(tbl_death)



In [50]:
ch_death <- ch_death %>% mutate(date_death = ifelse(is.na(dod), reg_date,dod))

In [51]:
ch_death$date_death <- as_date(ch_death$date_death)

In [52]:
## join death certificate tibble to the deathdate in the master cohort table 

In [53]:
ch_death_filter <- ch_death %>% select(person_id,date_death)
sql_data_above_sixtyfive <- left_join(sql_data_above_sixtyfive,ch_death_filter,by='person_id')

In [54]:
sql_data_above_sixtyfive <- sql_data_above_sixtyfive %>% mutate(dod = coalesce(sql_data_above_sixtyfive$death_date,sql_data_above_sixtyfive$date_death))%>% select(-death_date,-date_death)


In [55]:
sql_data_above_sixtyfive <- sql_data_above_sixtyfive %>% mutate(mortality = dod-episodestartdate)

In [56]:
alive_morethan_6weeks <- sql_data_above_sixtyfive %>% filter((mortality > 42) |is.na(dod))

In [57]:
alive_morethan_6weeks %>% n_distinct()

[1] 2945

In [58]:
##next we remove anyone who was discharged within 42 days or died within 42 days
#first we calculate the number and then remove anyone who died within 42 days 
long_stay_resident <- alive_morethan_6weeks %>% filter(episodestopdate > (episodestartdate + ddays(x=42)))


In [59]:
#second, of  those remaining, we calculate and remove those who had a recorded legnth of stay of 42 days or less. 
#the two figures combined (number died withi 42 days and number discharged alive) gives the total number filtered at this stage. 

In [60]:
long_stay_resident %>% n_distinct()

[1] 2884

In [61]:
# we iidentified anyone with missing primary care data and removed them, 
# this is most likely because whilst they are recieving social care funding from the bradford local authorty they are not registered to a GP contributing to Connected Bradford. 

In [62]:
sql_missing <- paste('
select
distinct b.person_id as primary_care_person
from `yhcr-prd-bradfor-bia-core.CB_FDM_AdultSocialCare.person` b
left join (select person_id,
  from `yhcr-prd-bradfor-bia-core.CB_FDM_PrimaryCare.tbl_srcode`
  where person_id in(select person_id from `yhcr-prd-bradfor-bia-core.CB_FDM_AdultSocialCare.person`)) a
  on b.person_id = a.person_id
  where a.person_id is null

                     ;',sep="")

tbl_missing <-bq_project_query(ProjectId,sql_missing)
tbl_missing <- bq_table_download(tbl_missing)


Warning message in as.integer.integer64(x):
“NAs produced by integer overflow”


In [63]:
care_home_cohort_final <- long_stay_resident %>% filter(!(long_stay_resident$person_id %in% tbl_missing$primary_care_person)) 
care_home_cohort_final %>%  n_distinct()
care_home_ghosts <- care_home_cohort_final %>% filter( is.na(dod))

[1] 2793

the 'ghosts' are those who had a missing death date 

In [64]:
care_home_ghosts %>% n_distinct()

[1] 147

In [65]:
#send the final care home cohort back to bigquery as a table so it can be ran through the other queries
care_home_cohort_final$mortality <- as.numeric(care_home_cohort_final$mortality)

In [67]:
mybq = bq_table(project='yhcr-prd-bradfor-bia-core', dataset='CB_2172', table='care_home_cohort_v1')
bq_table_upload(x=mybq, values= care_home_cohort_final, create_disposition='CREATE_IF_NEEDED', 
             write_disposition='WRITE_TRUNCATE')

In [68]:
sql_ch <- paste('
with a as (
select
* except(first_episodestartdate,finalisedPSR,episodestartdate,episodestopdate), 
cast(first_episodestartdate as DATETIME) as first_episodestartdate
from ',targetdb1,'.care_home_cohort_v1)

select
a.*,
b.observation_period_end_date
from a
left join `yhcr-prd-bradfor-bia-core.CB_FDM_PrimaryCare.observation_period` b
on a.person_id = b.person_id

                     ;',sep="")

tbl_ch <-bq_project_query(ProjectId,sql_ch)
tbl_ch <- bq_table_download(tbl_ch)



this section merges the data from 

In [69]:
tbl_ch_dod_observation_merged <- tbl_ch %>% mutate(dod = coalesce(tbl_ch$dod,tbl_ch$observation_period_end_date))
